In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

In [2]:
class MyConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size) * 0.1)
        self.bias=  nn.Parameter(torch.zeros(out_channels))
        self.stride = stride
        self.padding = padding

    def forward(self, x):
        return nn.functional.conv2d(x, self.weight, self.bias, stride=self.stride, padding=self.padding)

In [25]:
class HorseColorDataset(Dataset):
    def __init__(self, images, kmeans_colors):
        self.kmeans_colors = kmeans_colors
        self.horse_images = images
        self.horse_images = self.horse_images.float() / 255.0
        self.gray_images = 0.299*self.horse_images[:,0,:,:] + 0.587*self.horse_images[:,1,:,:] + 0.114*self.horse_images[:,2,:,:]
        self.gray_images = self.gray_images.unsqueeze(1)
        self.labels = self.rgb_to_class(self.horse_images)
    
    def rgb_to_class(self, imgs):
        """
        Map each pixel to nearest k-means cluster index
        """
        N, C, H, W = imgs.shape
        labels = torch.zeros(N, H, W, dtype=torch.long)
        kmeans = torch.tensor(self.kmeans_colors, dtype=torch.float32)  # [24,3]
        for i in range(N):
            pix = imgs[i].permute(1,2,0).reshape(-1,3)  # [H*W,3]
            # compute distance to each cluster
            dists = torch.cdist(pix, kmeans)  # [H*W,24]
            labels[i] = torch.argmin(dists, dim=1).reshape(H,W)
        return labels

    def __len__(self):
        return len(self.horse_images)

    def __getitem__(self, idx):
        return self.gray_images[idx], self.labels[idx]

In [3]:
class ColorizationCNN_Classification(nn.Module):
    def __init__(self, num_classes=24):
        super().__init__()
        self.net = nn.Sequential(
            MyConv2d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            MyConv2d(64, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d(2),

            MyConv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
            MyConv2d(128, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128), nn.MaxPool2d(2),

            MyConv2d(128, 256, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(256),
            MyConv2d(256, 256, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(256),

            nn.Upsample(scale_factor=2, mode='nearest'),
            MyConv2d(256, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
            nn.Upsample(scale_factor=2, mode='nearest'),
            MyConv2d(128, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),

            # Output logits: 24 classes for each 32x32 pixel
            MyConv2d(64, num_classes, 1, padding=0)
        )

    def forward(self, x):
        return self.net(x)

In [4]:
# Sanity check: output logits should match label spatial size
x_dummy = torch.randn(2, 1, 32, 32)
with torch.no_grad():
    logits_dummy = ColorizationCNN_Classification(num_classes=24)(x_dummy)
print("Logit shape:", tuple(logits_dummy.shape))

Logit shape: (2, 24, 32, 32)


In [27]:
transform = transforms.ToTensor()
cifar10 = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
images = torch.stack([transform(img) for img, lbl in cifar10])  # [N,3,32,32]
labels = torch.tensor([lbl for img,lbl in cifar10])

horse_idx = (labels==7).nonzero(as_tuple=True)[0]
horse_images = images[horse_idx]

kmeans_colors = np.load("colour_kmeans24_cat7.npy", allow_pickle=True, encoding='latin1')

# Extract ONLY the color centers
kmeans_colors = kmeans_colors[0]

# Convert to proper dtype
kmeans_colors = np.array(kmeans_colors, dtype=np.float32)

print(kmeans_colors.shape)

dataset = HorseColorDataset(horse_images, kmeans_colors)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

(24, 3)


In [28]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ColorizationCNN_Classification().to(device)
criterion = nn.CrossEntropyLoss()  # pixel-wise classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):
    epoch_loss = 0
    for gray, label in loader:
        gray = gray.to(device)
        label = label.to(device)
        optimizer.zero_grad()
        out = model(gray)  # [B,24,H,W]
        loss = criterion(out, label)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(loader):.4f}")

RuntimeError: input and target batch or spatial sizes don't match: target [32, 32, 32], input [32, 24, 8, 8]

In [ ]:
model.eval()
gray, label = next(iter(loader))
gray = gray.to(device)
with torch.no_grad():
    logits = model(gray)  # [B,24,H,W]
    pred_classes = logits.argmax(dim=1)  # [B,H,W]

In [ ]:
# Map back to RGB
pred_rgb = torch.zeros(gray.shape[0],3,32,32)
for i in range(gray.shape[0]):
    pred_rgb[i] = torch.tensor(kmeans_colors[pred_classes[i].cpu()], dtype=torch.float32).permute(2,0,1)

# Show 1 example
plt.subplot(1,3,1)
plt.title("Grayscale Input")
plt.imshow(gray[0,0].cpu(), cmap="gray")
plt.subplot(1,3,2)
plt.title("Predicted Colors")
plt.imshow(pred_rgb[0].permute(1,2,0).cpu())
plt.subplot(1,3,3)
plt.title("Ground Truth")
plt.imshow(horse_images[0].permute(1,2,0))
plt.show()